# Stage 2: Subtraction
Getting rid of all identical objects throughout every single image to easily spot differences.

## Cel 1: Setup + load the aligend frames from disk

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import config  # loads .env -> $ZTFDATA, installs the np.in1d shim (import FIRST)

from ztfquery.io import LOCALSOURCE
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import matplotlib.pyplot as plt
import glob

# Load the aligned frames we saved (data + the shared WCS from each header)
aligned_dir = Path(LOCALSOURCE) / "aligned"
aligned_files = sorted(glob.glob(str(aligned_dir / "*sciimg.fits")))

frames = []  # list of (data, wcs, name)
for path in aligned_files:
    with fits.open(path) as hdul:
        data = hdul[0].data.astype(float)
        wcs = WCS(hdul[0].header)
    frames.append((data, wcs, Path(path).name))
    print(f"Loaded {Path(path).name}  shape={data.shape}")

print(f"\nLoaded {len(frames)} aligned frames.")


## Cell 2: find template and cut out a small patch
cutting out a small patch for testing purposes

In [ ]:
from operator import pos
from astropy.nddata import Cutout2D

#template from sharpest frame
#science cutout at same path as template measured against template
tmpl_data, tmpl_wcs, tmpl_name = frames[2]
sci_data, sci_wcs, sci_name = frames[0]

print(f"Template : {tmpl_name}")
print(f"Science : {sci_name}\n")

# Cutout 1000x1000 central window from all images put together and aligned
size = (1000, 1000)
center = (sci_data.shape[1] // 2, sci_data.shape[0] // 2)

sci_cut = Cutout2D(sci_data, position=center, size=size, wcs=sci_wcs)
tmpl_cut = Cutout2D(tmpl_data, position=center, size=size, wcs=tmpl_wcs)

sci = sci_cut.data
tmpl = tmpl_cut.data


print(f"Science cutout : {sci.shape}, NaNs={np.isnan(sci).sum()}")
print(f"Template cutout: {tmpl.shape}, NaNs={np.isnan(tmpl).sum()}")


## Cell 3: Rung 1: First subtraction without any form of matching
Control group of experiment. BTW it's meant to look bad, so don't worry if it does

In [ ]:
diff_naive = sci - tmpl

# Color limits — force to plain Python floats (avoids matplotlib's size-1-array ValueError)
finite = diff_naive[np.isfinite(diff_naive)]
lim = float(np.nanpercentile(np.abs(finite), 99))

vmin = float(np.nanpercentile(sci, 5))
vmax = float(np.nanpercentile(sci, 99))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(sci,  cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[0].set_title(f"Science\n{sci_name[:28]}", fontsize=8)

axes[1].imshow(tmpl, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
axes[1].set_title(f"Template (sharpest)\n{tmpl_name[:28]}", fontsize=8)

axes[2].imshow(diff_naive, cmap="gray", origin="lower", vmin=-lim, vmax=+lim)
axes[2].set_title("RUNG 1: science - template\n(look for DIPOLES)", fontsize=8)

plt.tight_layout()
plt.show()

baseline_std = float(np.nanstd(diff_naive))
print(f"Baseline residual std (Rung 1): {baseline_std:.3f}")
print("Lower = cleaner cancellation. ois (Rung 2) should drop this a lot.")


## Cell 4: install ois into this notebook's environment

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip setuptools wheel



In [ ]:
import sys
!{sys.executable} -m pip install --no-build-isolation "git+https://github.com/toros-astro/ois.git"


## Cell 5: get function-calling of ois imports

In [ ]:
import ois
print("ois version:", getattr(ois, "__version__", "unknown"))
print()
help(ois.optimal_system)

## Cell 6: RUNG 2: ois configuration
we're matching reference image on science variable's frame

In [ ]:
import ois

diff_ois, optimal_image, kernel, background = ois.optimal_system(
    image=sci,        # science frame (blurrier here)
    refimage=tmpl,    # template (sharpest) -> convolved to match science
    kernelshape=(11, 11),
    method='Bramich',
)

# Same stretch for both panels so the comparison is fair.
lim_naive = float(np.nanpercentile(np.abs(diff_naive[np.isfinite(diff_naive)]), 99))
lim_ois   = float(np.nanpercentile(np.abs(diff_ois[np.isfinite(diff_ois)]), 99))
lim = max(lim_naive, lim_ois)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(diff_naive, cmap="gray", origin="lower", vmin=-lim, vmax=+lim)
axes[0].set_title(f"RUNG 1: naive\nstd = {baseline_std:.1f}", fontsize=9)

ois_std = float(np.nanstd(diff_ois))
axes[1].imshow(diff_ois, cmap="gray", origin="lower", vmin=-lim, vmax=+lim)
axes[1].set_title(f"RUNG 2: ois PSF-matched\nstd = {ois_std:.1f}", fontsize=9)

plt.tight_layout()
plt.show()

print(f"Rung 1 (naive) std : {baseline_std:.3f}")
print(f"Rung 2 (ois)   std : {ois_std:.3f}")
print(f"Improvement        : {baseline_std - ois_std:.3f} lower  "
      f"({100*(baseline_std-ois_std)/baseline_std:.1f}% reduction)")


## TRYING SAME THING JUST A DIFFERENT WAY OF CONFIGURING with PYZOGY

## Cell 7: Rung 3: Finding stars and building them

In [ ]:
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder

#1. measure background noise of sky to remove it later on. 
mean, median, std = sigma_clipped_stats(sci, sigma=3.0)
print(f"sky background: mean={mean:.2f} median={median:.2f} std={std:.2f}")

#2. find stars
finder = DAOStarFinder(fwhm=3.0, threshold=5.0 * std)
sources_sci = finder(sci - median)

print(f"\nFound {len(sources_sci)} sources in the science image.")
print(sources_sci[:5])   # peek at the first few (columns: xcentroid, ycentroid, flux, ...)


In [ ]:
from photutils.aperture import CircularAperture

positions = np.transpose((sources_sci['x_centroid'], sources_sci['y_centroid']))
apertures = CircularAperture(positions, r=8.0)

vmin, vmax = float(np.nanpercentile(sci, 50)), float(np.nanpercentile(sci, 99.5))

plt.figure(figsize=(9,9))
plt.imshow(sci, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
apertures.plot(color="red", lw=1.0, alpha=0.7)
plt.title(f"{len(sources_sci)} DAOStarFinder detections (science frame)", fontsize=9)
plt.show()

## Cell 8: Take noise and filter it out, only leaving clean star cutouts

In [ ]:
from astropy.table import Table
from astropy.nddata import NDData
from photutils.psf import extract_stars

size = 25
half = size // 2
hi, wi = sci.shape

# Filter 1: drop stars too close to edge
x = sources_sci['x_centroid']
y = sources_sci['y_centroid']
edge_ok = ((x > half) & (x < wi - half) & (y > half) & (y < hi - half))

# Filter 2: drop saturated / stars that are too bright
peak = sources_sci['peak']
bright_ok = (peak > 50) & (peak < 0.7 * 60000)

keep = edge_ok & bright_ok
stars_tbl = Table()
stars_tbl['x'] = x[keep]
stars_tbl['y'] = y[keep]
print(f"Kept {len(stars_tbl)} of {len(sources_sci)} as PSF candidates "
      f"(edge + brightness filtered).")

#extracting cutouts now
nddata = NDData(data=sci-median)
stars = extract_stars(nddata, stars_tbl, size=size)
print(f"Extracted {len(stars)} clean star cutouts of size {size}x{size}.")

## Cell 9: Build the PSF (the EPSFBuilder Step)
Build matching filter for Rung 3

In [ ]:
from astropy.nddata import overlap_slices
from photutils.psf import EPSFBuilder

epsf_builder = EPSFBuilder(oversampling=2, maxiters=10, progress_bar=True)
epsf_sci, fitted_stars = epsf_builder(stars)

print(f"PSF model built. shape = {epsf_sci.data.shape}")

# visualization
plt.figure(figsize=(5,5))
plt.imshow(epsf_sci.data, origin="lower", cmap="viridis")
plt.colorbar(label="normalized intensity")
plt.title("Empirical PSF -- science frame", fontsize=10)
plt.show()

## Cell 10: build empirical PSF for the TEMPLATE

In [ ]:
# Stage 2 — Cell 9: RUNG 3a — build the empirical PSF for the TEMPLATE (same 4 steps)
from astropy.stats import sigma_clipped_stats
from astropy.table import Table
from astropy.nddata import NDData
from photutils.detection import DAOStarFinder
from photutils.psf import extract_stars, EPSFBuilder

# 1. Sky stats for the template
t_mean, t_median, t_std = sigma_clipped_stats(tmpl, sigma=3.0)
print(f"template sky: median={t_median:.2f}  std={t_std:.2f}")

# 2. Find stars in the template
t_finder = DAOStarFinder(fwhm=3.0, threshold=5.0 * t_std)
sources_tmpl = t_finder(tmpl - t_median)
print(f"found {len(sources_tmpl)} sources in the template")

# 3. Filter (edge + brightness) and extract — ALL from sources_tmpl
size = 25
half = size // 2
hi, wi = tmpl.shape
tx = sources_tmpl['x_centroid']
ty = sources_tmpl['y_centroid']
edge_ok   = (tx > half) & (tx < wi - half) & (ty > half) & (ty < hi - half)
bright_ok = (sources_tmpl['peak'] > 50) & (sources_tmpl['peak'] < 0.7 * 60000)
keep = edge_ok & bright_ok
t_tbl = Table()
t_tbl['x'] = tx[keep]
t_tbl['y'] = ty[keep]
print(f"kept {len(t_tbl)} PSF candidates")

t_nddata = NDData(data=tmpl - t_median)
t_stars = extract_stars(t_nddata, t_tbl, size=size)
print(f"extracted {len(t_stars)} clean cutouts")

# 4. Build the template PSF (slow cell — let the progress bar finish)
epsf_tmpl, _ = EPSFBuilder(oversampling=2, maxiters=10, progress_bar=True)(t_stars)
print(f"template PSF built. shape = {epsf_tmpl.data.shape}")

# Compare the two PSFs side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(epsf_sci.data,  origin="lower", cmap="viridis")
axes[0].set_title("Science PSF (seeing 1.856)", fontsize=9)
axes[1].imshow(epsf_tmpl.data, origin="lower", cmap="viridis")
axes[1].set_title("Template PSF (seeing 1.752, sharper)", fontsize=9)
plt.tight_layout()
plt.show()


##  Cell 11: Install and configure PyZOGY from GitHub

In [ ]:
# Install PyZOGY but skip the broken console-script entry point.
# --use-pep517 + installing the package's files without the script wrapper.
import sys
!{sys.executable} -m pip install --no-build-isolation --no-deps \
    "git+https://github.com/dguevel/PyZOGY.git" \
    --config-settings="--build-option=--no-user-cfg" 2>&1 | tail -5


## Cell 12: actually run PyZOGY and hope it works

In [ ]:
from PyZOGY import subtract
import numpy as np

#run the actual subtraction
diff_zogy = subtract.run_subtraction(
    science_image=sci,
    reference_image=tmpl,
    science_psf=epsf_sci.data,
    reference_psf=epsf_tmpl.data,
    science_saturation=50000,
    reference_saturation=50000,
    normalization='reference',
    show=False,
)

#show the results
if isinstance(diff_zogy, tuple):
    print("returned a tuple of", len(diff_zogy), "items; using item [0] as the difference")
    diff_zogy = diff_zogy[0]

diff_zogy = np.asarray(diff_zogy, dtype=float)
zogy_std = float(np.nanstd(diff_zogy))
print(f"ZOGY difference computed. shape={diff_zogy.shape}, std={zogy_std:.3f}")

## Cell 13: visually compare all 3 methods (each on its own stretch)

In [ ]:
# Stage 2 — Cell 11: visually compare all THREE difference images (each on its OWN stretch)
# Safety: recompute any difference that isn't in memory (e.g. after a kernel restart).
if 'diff_naive' not in globals():
    diff_naive = sci - tmpl
    print("recomputed diff_naive")
baseline_std = float(np.nanstd(diff_naive))
ois_std  = float(np.nanstd(diff_ois))
zogy_std = float(np.nanstd(diff_zogy))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
panels = [
    (diff_naive, f"RUNG 1: naive\nstd={baseline_std:.0f} (raw counts)"),
    (diff_ois,   f"RUNG 2: ois\nstd={ois_std:.0f} (raw counts)"),
    (diff_zogy,  f"RUNG 3: ZOGY\nstd={zogy_std:.0f} (whitened scale)"),
]

for ax, (d, title) in zip(axes, panels):
    finite = d[np.isfinite(d)]
    lim = float(np.nanpercentile(np.abs(finite), 99))   # each panel scaled to ITSELF
    ax.imshow(d, cmap="gray", origin="lower", vmin=-lim, vmax=+lim)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

print("Judge by EYE, not the std numbers (different units):")
print("  flat gray, stars GONE        = good cancellation")
print("  dipoles / bright cores at stars = leftover mismatch")
